# Liver Damage Simulation of Napa (Paracetamol) Overdose
### A PK-PD Compartmental Model with Monte Carlo Variations and Statistical Hypothesis Testing
**Course:** CSE 413: Simulation & Modeling Lab - Section 7B  
**Team:** K.G.B.P  
**Authors:** Mehedi Hassan Bhuiyan (0432320005101080), Md. Masud Rahman (0432320005101064), Hasib Al Mahmud Siddique (0432320005101095), Rudro Antony Mrong (0432320005101059)


## 1. Environment Setup and Library Imports


In [ ]:
import numpy as np
import scipy
from scipy.integrate import solve_ivp
from scipy import stats
import matplotlib.pyplot as plt
import pandas as pd

# Import from our modular simulation package
from napa_simulation.models.formulations import Formulation, FormulationType
from napa_simulation.simulation.engine import simulate_patient, simulate_cohort
from napa_simulation.analysis.hypothesis_testing import (
    run_experiment_1_acute_vs_staggered,
    run_experiment_2_malnutrition,
    run_statistical_test_suite
)
from napa_simulation.visualization.plots import plot_timeline_multipanel, plot_cohort_distributions

print("All simulation modules successfully imported!")


## 2. Deterministic Single-Patient Trajectories
We evaluate the pharmacokinetics and toxicodynamics across three standard Bangladeshi Napa formulations:
1. Standard Napa (500 mg immediate release)
2. Napa Extend (665 mg bi-layer modified release: 36% immediate, 64% sustained)
3. Napa Extra (500 mg APAP + 65 mg Caffeine; caffeine induces CYP enzymes by +22%)
4. Acute Overdose (10,000 mg APAP = 20 tablets)


In [ ]:
# Run simulations
res_std = simulate_patient(formulation=Formulation.standard_napa(tablets=2))      # Therapeutic 1,000 mg
res_ext = simulate_patient(formulation=Formulation.napa_extend(tablets=2))        # Therapeutic 1,330 mg
res_extra = simulate_patient(formulation=Formulation.napa_extra(tablets=2))      # Therapeutic 1,000 mg + Caffeine
res_overdose = simulate_patient(formulation=Formulation.standard_napa(tablets=20)) # Overdose 10,000 mg

# Plot 4-panel timeline comparison
fig = plot_timeline_multipanel({
    "Standard Napa (1,000 mg)": res_std,
    "Napa Extend (1,330 mg)": res_ext,
    "Napa Extra (1,000 mg + Caff)": res_extra,
    "Acute Overdose (10,000 mg)": res_overdose,
}, title="Figure 1: Pharmacokinetics & Toxicodynamics Across Napa Formulations")

plt.show()


## 3. High-Throughput Monte Carlo Cohort Simulation
Deterministic ODEs produce a single static curve. To model biological reality, we inject stochastic human variation:
- Central Distribution Volume: V_d ~ N(50, 5) L
- Renal Clearance: k_el ~ N(0.035, 0.005) h^-1
- Baseline Glutathione: Well-Nourished ~ N(100, 7.5)% vs Malnourished ~ N(50, 7.0)%


In [ ]:
# Simulate virtual cohorts (N=100 patients per group)
cohort_acute = simulate_cohort(n_patients=100, dose_mg=10000.0, formulation_type=FormulationType.STANDARD, patient_type="normal", random_seed=42)
cohort_malnourished = simulate_cohort(n_patients=100, dose_mg=5000.0, formulation_type=FormulationType.STANDARD, patient_type="malnourished", random_seed=43)
cohort_normal = simulate_cohort(n_patients=100, dose_mg=5000.0, formulation_type=FormulationType.STANDARD, patient_type="normal", random_seed=44)

# Plot violin & box distributions of necrosis
fig_dist = plot_cohort_distributions({
    "Acute Overdose (10g)": cohort_acute,
    "Malnourished (5g)": cohort_malnourished,
    "Well-Nourished (5g)": cohort_normal,
}, title="Figure 2: Monte Carlo Necrosis Distributions Across Cohorts")

plt.show()


## 4. Automated Statistical Hypothesis Testing Suite
We test our two primary clinical damage hypotheses:
- **Experiment 1:** Acute Bolus vs. Staggered Ingestion (10,000 mg APAP)
- **Experiment 2:** Nutritional Vulnerability at Borderline Dosing (5,000 mg APAP)


In [ ]:
# Run both experiments
test_results, summary_table = run_statistical_test_suite(n_patients=100, alpha=0.05)

# Display consolidated table
print(summary_table[["Experiment", "H0", "t-Statistic", "p-Value (t-test)", "Cohen's d", "Verdict"]].to_string(index=False))

# Detailed reports for each experiment
for r in test_results:
    print(r.format_defense_report())
    print()


## 5. Interactive Dosing Simulator (In-Notebook Sliders)
Using ipywidgets to dynamically adjust dosage and observe liver cell necrosis in real-time.


In [ ]:
from ipywidgets import interact, IntSlider, Dropdown

def interactive_dosing(tablets=4, formulation_choice="Standard Napa (500mg)", patient_nutrition="Well-Nourished (100% GSH)"):
    form_type = FormulationType.EXTEND if "Extend" in formulation_choice else (FormulationType.EXTRA if "Extra" in formulation_choice else FormulationType.STANDARD)
    gsh_base = 50.0 if "Malnourished" in patient_nutrition else 100.0
    form = Formulation.from_type(form_type, tablets=tablets)
    res = simulate_patient(formulation=form, initial_gsh=gsh_base)
    
    print(f"Total APAP Dose: {res.total_dose_mg:,.0f} mg")
    print(f"Peak Plasma APAP: {res.peak_c_plasma:.2f} mg/L (Tmax: {res.t_max:.2f} h)")
    print(f"Minimum Glutathione Pool: {res.min_gsh:.1f}% (Critical threshold is 30%)")
    print(f"Final Hepatocyte Necrosis: {res.final_necrosis:.2f}%")
    print(f"Clinical Risk Assessment: {res.clinical_risk}")

interact(
    interactive_dosing,
    tablets=IntSlider(min=1, max=30, step=1, value=4, description="Tablets:"),
    formulation_choice=Dropdown(options=["Standard Napa (500mg)", "Napa Extend (665mg)", "Napa Extra (500mg + Caff)"], value="Standard Napa (500mg)", description="Brand:"),
    patient_nutrition=Dropdown(options=["Well-Nourished (100% GSH)", "Malnourished (50% GSH)"], value="Well-Nourished (100% GSH)", description="Nutrition:")
);
